# Train INR Currency Counter on Google Colab (Local Upload)

This notebook trains a **YOLOv8s** (small) model — the best speed/accuracy tradeoff for our ~1900-image, 7-class dataset. Expect **~45-60 min on a T4 GPU**.

### Speed optimizations vs. the old YOLOv8x setup:
- **YOLOv8s** (11M params) instead of YOLOv8x (68M) — 5-6× faster per epoch, same accuracy at this dataset scale
- **Batch 32** (fits easily with the smaller model) — fewer gradient steps per epoch
- **RAM caching** — eliminates disk I/O bottleneck for small datasets
- **AdamW + cosine LR** — converges faster than SGD on small datasets
- **50 epochs with patience 15** — more than enough for convergence

## Setup Instructions:
1. Upload your local `merged.zip` file directly to Google Colab by clicking the **Files** icon on the left sidebar and dragging the file in.
2. Wait until the **circular upload progress ring** at the bottom-left of the sidebar completes and disappears (so the file is fully uploaded).
3. Set your runtime type to GPU (**Runtime -> Change runtime type -> T4 GPU / L4 GPU**).
4. Run all cells in order.

## Step 1: Install Dependencies
Install the required libraries on the Colab environment:

In [ ]:
!pip install ultralytics torch torchvision opencv-python pyyaml

## Step 2: Verify GPU Acceleration
Verify that Google Colab has assigned a GPU to your runtime session:

In [ ]:
import torch
print(f"PyTorch version: {torch.__version__}")
print(f"CUDA (GPU) available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU Device: {torch.cuda.get_device_name(0)}")

## Step 3: Unzip Your Uploaded Dataset
Run this cell to extract the uploaded `merged.zip` dataset to the local directory. The `-o` flag ensures any existing files are overwritten, and `-q` runs it silently without showing thousands of files.

In [ ]:
import os

zip_path = '/content/merged.zip'
if os.path.exists(zip_path):
    # Create local data directory
    !mkdir -p /content/data
    
    # Unzip silently and overwrite existing
    !unzip -o -q /content/merged.zip -d /content/data/
    print("Dataset successfully extracted to /content/data!")
    !ls -la /content/data/
else:
    print('Error: Could not find /content/merged.zip.')
    print('Please ensure your merged.zip file has fully finished uploading to the files panel before running this cell.')


## Step 4: Fix and Verify the Dataset Config (data.yaml)
Since the dataset configuration references your local Windows paths, this step automatically rewrites the base path to Colab's Linux directory `/content/data`:

In [ ]:
import yaml

yaml_path = '/content/data/data.yaml'
try:
    with open(yaml_path, 'r') as f:
        cfg = yaml.safe_load(f)

    # Update base path for Google Colab Linux VM
    cfg['path'] = '/content/data'

    with open(yaml_path, 'w') as f:
        yaml.dump(cfg, f, sort_keys=False)
        
    print("data.yaml successfully updated for Colab:")
    with open(yaml_path, 'r') as f:
        print(f.read())
except Exception as e:
    print(f"Error updating data.yaml: {e}")

## Step 5: Download Base Model (YOLOv8s)
Download the pre-trained YOLOv8 Small weights from Ultralytics:

In [ ]:
!wget -q https://github.com/ultralytics/assets/releases/download/v8.2.0/yolov8s.pt -O /content/yolov8s.pt

## Step 6: Train the YOLOv8s Model
Train your model. This should take **~45-60 minutes on a T4 GPU** (down from 4.5+ hours with YOLOv8x).

In [ ]:
from ultralytics import YOLO

# Load YOLOv8s — best speed/accuracy for ~1900 images, 7 classes
model = YOLO('/content/yolov8s.pt')

# Train with speed-optimized settings
results = model.train(
    data='/content/data/data.yaml',
    epochs=50,             # Fewer epochs needed with AdamW + cosine LR
    imgsz=640,
    batch=32,              # YOLOv8s fits batch=32 easily on T4 (drop to 16 on OOM)
    workers=4,             # Colab has limited CPUs, 8 causes bottlenecks
    project='/content/models',
    name='inr_counter_s',
    cache='ram',           # Cache images in RAM — dataset is small enough (~1900 imgs)
    optimizer='AdamW',     # Converges much faster than default SGD on small datasets
    cos_lr=True,           # Cosine LR decay — better late-training convergence
    warmup_epochs=3,       # Shorter warmup
    amp=True,              # Mixed precision — faster on T4 Tensor Cores
    degrees=10.0,          # Random rotation range
    shear=2.0,
    perspective=0.0005,    # Perspective warp for camera angles
    mixup=0.15,            # Mixup augmentation for overlapping notes
    patience=15,           # Stop early if no improvement for 15 epochs
)

## Step 7: Download Your Trained Weights
This cell triggers an automatic browser download of your trained model (`best.pt`) to your local computer.

In [ ]:
from google.colab import files
files.download('/content/models/inr_counter_s/weights/best.pt')